🔹 Step 1: Environment Setup

In [164]:
# Install required libraries (run once)
# pip install pandas numpy matplotlib scikit-learn tensorflow

In [165]:
!python --version

Python 3.12.12


🔹 Step 2: Import Libraries

In [166]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

🔹 Step 3: Load & Inspect Dataset

In [167]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")
print(df.columns)

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [168]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [169]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [170]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [171]:
df.columns

Index(['2401', 'Borderlands', 'sentiment', 'text'], dtype='object')

In [172]:
df.sample(5)

,2401,Borderlands,sentiment,text
23232,4382,CS-GO,Irrelevant,[.. ESEA! ESEA! ESEA!. Good luck guys! Should ...
32319,7545,LeagueOfLegends,Negative,NaN
46505,11981,Verizon,Negative,A Verizon employee told me to lie on a claim. ...
53997,2077,CallOfDuty,Negative,"THATS of IT, NO IM YOU DONE AND WITH RANKED!!!..."
64083,7780,MaddenNFL,Negative,Digital RhandlerR this game so trash pic.twit...


In [173]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   2401         74681 non-null  int64 
 1   Borderlands  74681 non-null  object
 2   sentiment    74681 non-null  object
 3   text         73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [174]:
df = df[['text', 'sentiment']]

In [175]:
df.sample(5)

,text,sentiment
35175,"I had used to think Ellison was an ok guy, bec...",Neutral
73825,Fuck Microsoft and fuck Nvidia I'm losing my m...,Negative
33196,Dusty Jordan.,Neutral
69482,@TFSQ_LBvai<unk> luck.,Neutral
16455,I really wish a nigga sheriff would do some cr...,Irrelevant


In [176]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [177]:
df.shape

(74681, 2)

In [178]:
df.isnull().sum()

,0
text,686
sentiment,0


In [179]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [180]:
df.duplicated().sum()

np.int64(4909)

🔹 Step 4: Data Cleaning

In [181]:
df.dropna(inplace=True)

In [182]:
df.isnull().sum()

,0
text,0
sentiment,0


In [183]:
df['text'] = df['text'].astype(str)

In [184]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [185]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [186]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


🔹 Step 5: Encode Target Labels

In [187]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


In [188]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [189]:
df.sample(10)

,text,sentiment,sentiment_encoded
70926,walker is dead operation greenstone is over wh...,Neutral,2
9054,standing in line to watch and get an anubis te...,Neutral,2
9649,okay new years greetings play more games play ...,Positive,3
6152,jeff bezos should take down amazon and say fuc...,Negative,1
10132,lets the go,Positive,3
715,me omg i miss rhys soooo much i would do anyth...,Positive,3
55649,when you play your best but it doesnt count,Irrelevant,0
6058,time abby a premed major moved into the dorms ...,Neutral,2
11983,nk myteam please fix the pack coefficients,Negative,1
52336,youtubecomwatchvmeywv the dead redemption mov...,Neutral,2
